# Food Recipe Classification\n\nBinary classification of American and Italian recipes using TF-IDF ingredient features. The final model combines an RBF-SVM, Random Forest, and Extra Trees classifier using soft voting.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier, RandomForestClassifier, ExtraTreesClassifier

RANDOM_STATE = 2026

# Load data
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

X = train.drop(columns=["y"])
y = train["y"].astype(int)
X_test = test.copy()

print("Train shape:", X.shape)
print("Test shape:", X_test.shape)
print("\nClass distribution:")
print(y.value_counts().sort_index())

# Stratified 10-fold cross-validation
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=RANDOM_STATE
)

# Models
svm = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(
        kernel="rbf",
        C=3.5,
        gamma=0.04,
        probability=True,
        random_state=RANDOM_STATE
    ))
])

rf = RandomForestClassifier(
    n_estimators=500,
    min_samples_leaf=1,
    max_features="sqrt",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

et = ExtraTreesClassifier(
    n_estimators=500,
    min_samples_leaf=1,
    max_features="sqrt",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

# Evaluate individual models
models = {
    "SVM": svm,
    "Random Forest": rf,
    "Extra Trees": et
}

print("\nIndividual model CV results:")
for name, model in models.items():
    scores = cross_val_score(
        model, X, y, cv=cv, scoring="accuracy", n_jobs=-1
    )
    estimated_errors = (1 - scores.mean()) * len(X_test)
    print(
        f"{name:20s} accuracy = {scores.mean():.4f} "
        f"+/- {scores.std():.4f} | est. errors = {estimated_errors:.0f}"
    )

# Final soft-voting ensemble
final_model = VotingClassifier(
    estimators=[
        ("svm", svm),
        ("rf", rf),
        ("et", et)
    ],
    voting="soft",
    n_jobs=1
)

# Evaluate final model
print("\nFinal ensemble CV:")
scores = cross_val_score(
    final_model, X, y, cv=cv, scoring="accuracy", n_jobs=1
)

estimated_errors = (1 - scores.mean()) * len(X_test)

print(
    f"Voting SVM + RF + ET: {scores.mean():.4f} "
    f"+/- {scores.std():.4f}"
)
print(
    f"Estimated errors on {len(X_test)} observations: "
    f"~{estimated_errors:.0f}"
)

# Train on all labeled data
print("\nFitting final model on full training data...")
final_model.fit(X, y)

# Predict test set
pred = final_model.predict(X_test).astype(int)

# Safety checks
assert len(pred) == len(X_test)
assert set(np.unique(pred)).issubset({1, 2})

# Save submission
np.savetxt("y_pred.txt", pred, fmt="%d")

print("\nSaved y_pred.txt")
print("\nPrediction distribution:")
print(pd.Series(pred).value_counts().sort_index())
print("\nAll checks passed.")
